# SAC Collector — Plan A 4-tier Collection

**目标**：执行 [`docs/arrival_v2_sac_collector_design.md`](../docs/arrival_v2_sac_collector_design.md) **rev.3 §4.0 Plan A** — 从 `cross_u10_regression/.../seed_46` 训练过程切片 4 个 step ckpt × 1000 ep × stochastic 收集，落出真 D4RL `random / medium / medium_expert / expert` 四档 dataset。

**前置 commits（已闭环，不需要再 audit）**：
- `97d394c` — `SACCheckpointPolicy` adapter + `collect_offline_data.py` SAC mode（5 tests pass）
- `be4b573` + `8667fbc` — D4RL tier audit notebook（auto-discover 39 ckpt）
- `9f8252e` — audit completed1 实验记录（dense sweep verdict `✅ GO`）
- `e0d3544` — docs rev.3 + offline summary §4.3

**Collection mode 决策（本 session 已选）**：
- **Step 1 = A**：4 tier 全 stochastic（不加 `--sac-deterministic` flag）
  - rationale：与 D4RL `medium` / `medium-replay` 范式一致；`next_actions` 真随机更适合 BC penalty 算法（FQL / ReBRAC β1 sweep）；与 audit 阶段 stochastic eval 数字可直接对比。
- **Step 2 = B**：`replay_latest.pkl` (412 MB) 第 5 档 `medium-replay` **不在本 notebook 做** — 4 tier 闭环 + FQL/ReBRAC head-to-head 之后再启动 Plan B。

**预算**：1000 ep × 4 tier × `--num-workers 8` ≈ **30 min L4 CPU pool**（无 GPU 加速；SAC actor 在 worker 内 CPU forward）。

**Dataset 输出**（spec §4.0.4 锁定命名）：
```
offline_data/sac_random_s0_h4_arrival_v2_re150_u10cross_seed46_step25k_ep1000/
offline_data/sac_medium_s0_h4_arrival_v2_re150_u10cross_seed46_step425k_ep1000/
offline_data/sac_mexp_s0_h4_arrival_v2_re150_u10cross_seed46_step575k_ep1000/
offline_data/sac_expert_s0_h4_arrival_v2_re150_u10cross_seed46_step600k_ep1000/
```

## 0. 环境检查

In [ ]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}  (collection 用 CPU pool，CUDA 仅供 sanity)")
if torch.cuda.is_available():
    print(f"Device name:    {torch.cuda.get_device_name(0)}")

## 1. 挂载 Drive + cd 到项目根

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd

## 2. Sanity check — 4 ckpt + flow file + trainer_state.json

**目的**：在跑 4 个长 collect 命令前确认所有文件路径正确，避免 30 min 后才发现 ckpt 路径打错。

**检查**：
1. 4 个 `agent_step_*.pt` ckpt 存在（Drive 路径 = spec §4.0.1 锁定）
2. flow file `wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy` 存在
3. `trainer_state.json` 在 `experiments/<...>/seed_46/` 存在（Layer-2 sanity autodetect 依赖）

**预期**：所有 6 项 `✅ exists`。任何一项 `⚠️` 都需停下来定位（不要跳过 sanity 直接开跑）。

In [ ]:
from pathlib import Path

CKPT_DIR = Path("checkpoints/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46")
EXP_DIR  = Path("experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46")
FLOW     = Path("wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy")

# 4 tier ckpt（spec §4.0.1 锁定）
TIERS = [
    ("random",        25_002,  "agent_step_00025002.pt",  "step25k"),
    ("medium",        425_004, "agent_step_00425004.pt",  "step425k"),
    ("mexp",          575_004, "agent_step_00575004.pt",  "step575k"),
    ("expert",        600_000, "agent_step_00600000.pt",  "step600k"),
]

all_ok = True
for tier, step, ckpt_name, _ in TIERS:
    p = CKPT_DIR / ckpt_name
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  ✅ {tier:10s} ({step:>7d})  {p}  ({size_mb:.1f} MB)")
    else:
        print(f"  ⚠️ {tier:10s} ({step:>7d})  {p}  NOT FOUND")
        all_ok = False

print()
if FLOW.exists():
    print(f"  ✅ flow      {FLOW}  ({FLOW.stat().st_size/1e6:.1f} MB)")
else:
    print(f"  ⚠️ flow      {FLOW}  NOT FOUND")
    all_ok = False

ts_path = EXP_DIR / "trainer_state.json"
if ts_path.exists():
    print(f"  ✅ trainer_state.json  {ts_path}")
else:
    print(f"  ⚠️ trainer_state.json  {ts_path}  NOT FOUND")
    all_ok = False

print()
if all_ok:
    print("✅ Sanity check passed — 可以开始 collection.")
else:
    raise FileNotFoundError("Sanity check failed — 见上面 ⚠️ 项，定位后再 rerun.")

## 3. 协议参数（与 ckpt 训练 reset_options 严格一致）

**来源**：spec §4.0.3 协议表（来自 audit notebook cell 9 读出的 `trainer_state.json`）。

| 项 | 值 |
|---|---|
| probe-layout | `s0` |
| history-length | `4` |
| task-geometry | `cross_stream` |
| target-speed | `1.5` |
| objective | `arrival_v2` |
| flow | `wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy` |
| episodes | `1000` |
| num-workers | `8` |
| seed (collection) | `0`（collection env seed；与训练 seed 46 区分，与现有 `crosscomp_*` dataset 对齐） |
| sac-device | `cpu` |
| **sac-deterministic** | **OFF**（Mode A：全 stochastic） |

**Layer-2 sanity**：不显式传 `--sac-trainer-state` → collector autodetect 走 `checkpoints/<X>` ↔ `experiments/<X>` 路径镜像（cell 2 已确认 `trainer_state.json` 存在）。

In [ ]:
import os

os.environ["PLAN_A_FLOW"] = str(FLOW)
os.environ["PLAN_A_EPISODES"] = "1000"
os.environ["PLAN_A_WORKERS"] = "8"
os.environ["PLAN_A_SEED"] = "0"
os.environ["PLAN_A_CKPT_DIR"] = str(CKPT_DIR)

for tier, step, ckpt_name, step_tag in TIERS:
    os.environ[f"CKPT_{tier.upper()}"] = str(CKPT_DIR / ckpt_name)
    os.environ[f"OUT_{tier.upper()}"]  = f"offline_data/sac_{tier}_s0_h4_arrival_v2_re150_u10cross_seed46_{step_tag}_ep1000"

print("env vars set:")
for k in sorted(os.environ):
    if k.startswith(("PLAN_A_", "CKPT_", "OUT_")):
        print(f"  {k:20s}  {os.environ[k]}")

## 4. Collect tier 1 — `random` (step=25_002, success=0%)

**预期输出**：`offline_data/sac_random_s0_h4_arrival_v2_re150_u10cross_seed46_step25k_ep1000/{transitions.npz, metadata.json}`

**预算**：~7-8 min L4 CPU pool（1000 ep × `--num-workers 8`，actor stochastic forward 比 rule-based baseline 略慢）。

**数据特征预期**：100% OOB 撞墙 → 长 reward tail 全负、`dones` 几乎全是 OOB termination。

In [ ]:
!python -m scripts.collect_offline_data \
    --sac-ckpt "$CKPT_RANDOM" \
    --probe-layout s0 \
    --history-length 4 \
    --task-geometry cross_stream \
    --target-speed 1.5 \
    --objective arrival_v2 \
    --flow "$PLAN_A_FLOW" \
    --episodes $PLAN_A_EPISODES \
    --num-workers $PLAN_A_WORKERS \
    --seed $PLAN_A_SEED \
    --sac-device cpu \
    --output-dir "$OUT_RANDOM"

## 5. Collect tier 2 — `medium` (step=425_004, success=50%)

**数据特征预期**：50% success / 30% OOB / 20% timeout — D4RL `medium` 经典 mixed-quality 范式。

In [ ]:
!python -m scripts.collect_offline_data \
    --sac-ckpt "$CKPT_MEDIUM" \
    --probe-layout s0 \
    --history-length 4 \
    --task-geometry cross_stream \
    --target-speed 1.5 \
    --objective arrival_v2 \
    --flow "$PLAN_A_FLOW" \
    --episodes $PLAN_A_EPISODES \
    --num-workers $PLAN_A_WORKERS \
    --seed $PLAN_A_SEED \
    --sac-device cpu \
    --output-dir "$OUT_MEDIUM"

## 6. Collect tier 3 — `medium_expert` (step=575_004, success=80%)

**数据特征预期**：80% success / 10% OOB / 10% timeout — D4RL `medium-expert` 准 expert 风格，但仍含少量 failure mode（适合测 BC penalty 的鲁棒性）。

In [ ]:
!python -m scripts.collect_offline_data \
    --sac-ckpt "$CKPT_MEXP" \
    --probe-layout s0 \
    --history-length 4 \
    --task-geometry cross_stream \
    --target-speed 1.5 \
    --objective arrival_v2 \
    --flow "$PLAN_A_FLOW" \
    --episodes $PLAN_A_EPISODES \
    --num-workers $PLAN_A_WORKERS \
    --seed $PLAN_A_SEED \
    --sac-device cpu \
    --output-dir "$OUT_MEXP"

## 7. Collect tier 4 — `expert` (step=600_000, success=100%)

**数据特征预期**：100% success / 0% OOB / 0% timeout — D4RL `expert` 干净 trajectory。`actions` 因 stochastic（Mode A）会有 σ_act 噪声，与 D4RL 经典 `expert`（deterministic）略不同；对 FQL / ReBRAC 而言相当于天然 action noise，更接近实际 deployment behavior。

**注意**：本档是 paper 1 / FQL P2 主对照所用的 sister dataset — 与现有 `crosscomp_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000`（rule-based crosscomp）形成 SAC-vs-rule head-to-head 第四轴。

In [ ]:
!python -m scripts.collect_offline_data \
    --sac-ckpt "$CKPT_EXPERT" \
    --probe-layout s0 \
    --history-length 4 \
    --task-geometry cross_stream \
    --target-speed 1.5 \
    --objective arrival_v2 \
    --flow "$PLAN_A_FLOW" \
    --episodes $PLAN_A_EPISODES \
    --num-workers $PLAN_A_WORKERS \
    --seed $PLAN_A_SEED \
    --sac-device cpu \
    --output-dir "$OUT_EXPERT"

## 8. Verify — 4 dataset 大小 / shape / metadata

**目的**：4 个 collect 完成后立刻确认
1. `transitions.npz` 存在 + episodes count 接近 1000（OOB tier 因 early-termination 可能略 < 1000 transitions 总数比 expert tier 少很多）
2. `metadata.json` 的 `behavior_source == "sac_checkpoint"` + `sac_ckpt_path` 正确
3. `obs.shape[1] == 48`（s0 + h=4 with arrival_v2 context channels）
4. `privileged_obs` 缺失（s0 vanilla SAC 无 asymmetric critic）

In [ ]:
import json
import numpy as np

for tier, step, ckpt_name, step_tag in TIERS:
    out_dir = Path(f"offline_data/sac_{tier}_s0_h4_arrival_v2_re150_u10cross_seed46_{step_tag}_ep1000")
    npz_path = out_dir / "transitions.npz"
    meta_path = out_dir / "metadata.json"

    print(f"=== tier={tier:10s}  step={step:>7d} ===")

    if not npz_path.exists():
        print(f"  ⚠️ {npz_path} NOT FOUND")
        continue

    with np.load(npz_path) as data:
        obs_shape = data["obs"].shape
        n_trans = obs_shape[0]
        has_priv = "privileged_obs" in data.files
        n_done = int(data["dones"].sum())
        mean_reward = float(data["rewards"].mean())

    meta = json.loads(meta_path.read_text())
    behavior_source = meta.get("behavior_source", "?")
    sac_ckpt = meta.get("sac_ckpt_path", "?")
    sac_det = meta.get("sac_deterministic", "?")

    print(f"  npz size:           {npz_path.stat().st_size/1e6:.1f} MB")
    print(f"  obs.shape:          {obs_shape}")
    print(f"  n_transitions:      {n_trans}")
    print(f"  n_dones (~episodes):{n_done}")
    print(f"  mean_reward:        {mean_reward:+.4f}")
    print(f"  has privileged_obs: {has_priv}  (expect False for s0 vanilla)")
    print(f"  behavior_source:    {behavior_source}  (expect 'sac_checkpoint')")
    print(f"  sac_deterministic:  {sac_det}  (expect False — Mode A)")
    print(f"  sac_ckpt_path:      .../{Path(sac_ckpt).name}")
    print()

## 9. Next steps（4 tier 闭环之后）

1. **Plan B medium-replay**（spec §4.0.6） — `replay_latest.pkl` (412 MB) → npz 第 5 档。写 `scripts/replay_to_offline.py`（~1-2h）。
2. **FQL + ReBRAC β1∈{1.0, 4.0} head-to-head × 4 tier × 2 seed = 16 run**（~4-6h L4）— 落出 paper 1 / FQL P2 SAC-collector 第四轴主结论。
3. **Multi-seed 扩展**（如 reviewer push back） — 按 spec §4.0.4 / rev.2 §4.2 模板补 `seed=47/50` 各 1 个 600k arrival_v2 SAC 训练 + 重收 4 tier。

**结果回收**：4 dataset 不 commit 进 git（gitignore），只把本 notebook 跑完结果（cell 8 verify 输出 + cell 4-7 collector 日志） commit 进 `_completed.ipynb` 副本作实验记录。